In [ ]:
%load_ext autoreload
%autoreload 2
import sys
DIANNE = ".."
sys.path.insert(0, f"{DIANNE}/dianne-core")
sys.path.insert(0, f"{DIANNE}/dianne-utils")
sys.path.insert(0, f"{DIANNE}/dianne-viewer")
import dianne_core
import dianne_utils as dianne
import dianne_viewer as viewer
dianne.setNotebookWidth(100)

In [ ]:
# Download the demo data from Zenodo and unzip it to the data folder
dianne.downloadZIPFromZenodo('../../data/', 'https://zenodo.org/records/19225051/files/', 'RMS2194.oid0.zip')

In [ ]:
# Load the data and prepare the patches for annotation and training the classifier
dataPath = '../../data/'
samples  = ['RMS2194.oid0']
classifierPaths = 'classifiers/'
dianne.setupClassifierPaths(classifierPaths)
fname = 'features/false-1-ctranspath_features.tsv.gz'
ads, imgs, patchCoordinates, patchesCDFs, qs, ts, mpp, L, N = dianne_core.loadDataAndPreparePatches(samples, dataPath, fname=fname, L=None, ts=112, mpp=0.25, N=4)

In [ ]:
# Display representative patches that can be used to start the annotation process
startParams = {}
dianne.jumpStart(patchCoordinates, patchesCDFs, imgs, startParams, ncols=6, nrows=3, pyramidscale=2,
                 ads=ads, L=1 if L is None else L, sh=int(ts/2)/mpp, figsize=(3, 3), seed=1, maxN=10**3)

In [ ]:
# Initialize the classifier and the annotation log
clf, plog, bp = {}, [], {}
try: bp.update({startParams['selected_patch']: 'positive'})
except: pass

In [ ]:
# Load the classifier if it exists, otherwise initialize empty classifier
clf, bp, plog, startParams = dianne.loadClassifier(classifierPaths, 'classifier-tumor')

In [ ]:
# Run the annotation loop with active learning and probabilistic patch selection
clf, plog, bp = {}, [], {}
try: bp.update({startParams['selected_patch']: 'positive'})
except: pass
dianne.runAnnotation(patchCoordinates, patchesCDFs, imgs, bp, clf, plog, ads=ads, qs=qs, minN=1, pyramidscale=2, 
            figsize=(5, 5), augFunc=dianne_core.PCMA, alpha=0.5, R=1, cmapColors=['lightcoral', 'blue'],
            seed=0, randomness=0.5, L=1 if L is None else L, sh=int(ts/2)/mpp)

In [ ]:
# After annotating at least 10 positive and 10 negative patches, we can inspect the annotated patches
dianne.inspectAnnotatedPatches(patchCoordinates, patchesCDFs, imgs, bp, augFunc=dianne_core.PCMA, alpha=0.5, nRepeats=25, pyramidscale=2,
                            L=1 if L is None else L, sh=int(ts/2)/mpp, verbose=False)

In [ ]:
# Save the classifier, the annotation log, and the parameters used for training the classifier
dianne.saveClassifier(clf, classifierPaths, 'classifier-tumor', dataPath, samples, patchesCDFs, L, ts, mpp, N, fname, qs, startParams, plog, bp)

In [ ]:
# Run inference on the entire slides
infSample = samples[0]
x, y, p = dianne_core.inferProbFast(ads[infSample], clf['clf'], qs, tsize=ts/mpp, R=2, erode=False)
xi, yi, pi = dianne.interpolatePoints(x, y, p, multiplier=16)
dianne.showProbImg(xi, yi, pi, f=2, figsize=(3, 3), ts=ts, mpp=mpp, title=infSample)

In [ ]:
# Create probability masks, extract contours and visualize them on the H&E image
downsampled_map, fshape = dianne.makeProbMask(ads[infSample], imgs[infSample], x, y, p, ts=ts, mpp=mpp, downfactor=16,
                                            savepath=classifierPaths, prefix=f'{infSample}-tumor', verbose=True)
geojson = dianne.extractContoursForQuPath(downsampled_map, fshape, cutoff=0.5, min_area=10**6, downfactor=16, sigma=100, savepath=classifierPaths, prefix=f'{infSample}-tumor')
dianne.viewContoursOnImage(imgs[infSample], geojson, fshape, level=1, figsize=(12, 12), linewidth=1)